# FastAPI + LLM Agent: From a Typed Request to a Validated Tool Call

**Question:** how does a Python function become an API that can safely execute a model-requested tool?

This worked example follows the request through each boundary. It uses the real application code, but substitutes scripted provider responses so the notebook is reproducible, offline, and free of API charges. Scripted responses prove control flow, not model quality.

| Layer | Responsibility | Evidence |
|---|---|---|
| Type hints | Describe inputs and outputs | Annotated models |
| Pydantic | Check structure and constraints | Accepted/rejected payloads |
| FastAPI | Expose HTTP and OpenAPI | TestClient responses |
| Agent loop | Exchange tool calls/results | Recorded call IDs |
| Tool dispatcher | Permit only validated operations | Invalid-call tests |

Run from the repository root. Install `requirements.txt` plus `nbformat nbclient ipykernel` to re-execute with `python execute_notebook.py`. No real secrets are needed.

In [1]:
import inspect, json
from types import SimpleNamespace
from unittest.mock import AsyncMock, patch
from IPython.display import Code, display
from fastapi.testclient import TestClient
from pydantic import ValidationError
from app.main import ChatRequest, create_app
from app.agent import LookupArguments, execute_tool, run_agent
print("Application imports successfully")

C:\Users\jacob\Documents\Codex\2026-09-26\inspect-my-existing-ai-growth-engine\outputs\AI Growth Engine\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Application imports successfully


## 1. Type hints become a runtime contract
`message: str` is a type hint. A plain Python function would not enforce it; `ChatRequest` inherits Pydantic behavior. FastAPI uses this model at the HTTP boundary. Constraints and strict validation reject empty or wrongly typed messages.

In [2]:
display(Code(inspect.getsource(ChatRequest), language="python"))
request = ChatRequest(message="  Explain tool calling  ")
assert request.message == "Explain tool calling"
for value in ["", "   ", 123]:
    try:
        ChatRequest(message=value)
    except ValidationError as error:
        print(repr(value), "rejected:", error.errors()[0]["type"])
print(request.model_dump())

class ChatRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True, strict=True)
    message: str = Field(min_length=1, max_length=2000)

'' rejected: string_too_short
'   ' rejected: string_too_short
123 rejected: string_type
{'message': 'Explain tool calling'}


## 2. Model output is another untrusted input
The tool accepts only three topics and rejects extra fields. A model requesting an unknown tool receives an error. This dispatcher cannot execute arbitrary code or change external records.

In [3]:
display(Code(inspect.getsource(LookupArguments), language="python"))
print(json.loads(execute_tool("lookup_lesson", '{"topic":"fastapi"}')))
assert "error" in json.loads(execute_tool("run_shell", '{}'))
assert "error" in json.loads(execute_tool("lookup_lesson", '{"topic":"secrets"}'))
print("Unknown tools and unsupported topics rejected")

class LookupArguments(BaseModel):
    model_config = ConfigDict(extra="forbid", strict=True)
    topic: Literal["fastapi", "pydantic", "tool_calling"]

{'topic': 'fastapi', 'lesson': 'FastAPI exposes Python functions as HTTP endpoints and generates OpenAPI documentation.', 'source': 'app/agent.py:LESSONS'}
Unknown tools and unsupported topics rejected


## 3. Trace the agent loop without making a provider call
The first scripted response asks for a tool; the second supplies a final answer. The real loop validates and executes the requested function, then returns its result under the same `call_id`. Test doubles exist only here and in tests; the running API has no fake-answer mode.

In [4]:
tool_call = SimpleNamespace(type="function_call", name="lookup_lesson",
                            arguments='{"topic":"pydantic"}', call_id="example_call_1")
create = AsyncMock(side_effect=[
    SimpleNamespace(status="completed", output=[tool_call], output_text=""),
    SimpleNamespace(status="completed", output=[], output_text="Scripted test answer: Pydantic validates typed data."),
])
provider = SimpleNamespace(responses=SimpleNamespace(create=create))
answer, used = await run_agent(provider, "test-model", "Explain Pydantic")
assert create.call_count == 2
assert used == ["lookup_lesson"]
result_item = create.call_args.kwargs["input"][-1]
assert result_item["call_id"] == "example_call_1"
print(json.dumps(result_item, indent=2))
print(answer)

{
  "type": "function_call_output",
  "call_id": "example_call_1",
  "output": "{\"topic\": \"pydantic\", \"lesson\": \"Pydantic validates incoming data using type hints and constraints. Type hints alone do not enforce runtime types.\", \"source\": \"app/agent.py:LESSONS\"}"
}
Scripted test answer: Pydantic validates typed data.


## 4. Exercise HTTP boundaries
TestClient calls the actual FastAPI application in process. Environment values below are explicitly fake and restored when the context exits. The agent is replaced only in this test, so no OpenAI request occurs. A missing token returns 401, invalid input 422, and a valid request 200.

In [5]:
with patch.dict("os.environ", {"OPENAI_API_KEY": "test-only", "OPENAI_MODEL": "test-model", "APP_API_TOKEN": "test-token"}), patch("app.main.run_agent", AsyncMock(return_value=("Scripted HTTP test answer", ["lookup_lesson"]))):
    with TestClient(create_app()) as client:
        auth = {"Authorization": "Bearer test-token"}
        statuses = {
            "health": client.get("/health").status_code,
            "no_token": client.post("/chat", json={"message": "hi"}).status_code,
            "invalid_body": client.post("/chat", headers=auth, json={"message": " "}).status_code,
            "valid_request": client.post("/chat", headers=auth, json={"message": "Explain FastAPI"}).status_code,
        }
assert statuses == {"health": 200, "no_token": 401, "invalid_body": 422, "valid_request": 200}
print(statuses)

{'health': 200, 'no_token': 401, 'invalid_body': 422, 'valid_request': 200}


## 5. Inspect the reusable implementation
Notice the loop bound, preserved output items, matching call IDs, and `store=False`. The `AsyncOpenAI` client is injected, making behavior testable without credentials. `list[Any]` is confined to heterogeneous provider protocol items; request and tool schemas are explicit.

In [6]:
display(Code(inspect.getsource(run_agent), language="python"))

async def run_agent(client: AsyncOpenAI, model: str, message: str) -> tuple[str, list[str]]:
    # Any is confined to heterogeneous provider protocol items.
    history: list[Any] = [{"role": "user", "content": message}]
    used: list[str] = []
    for _ in range(4):  # At most four provider requests per HTTP request.
        response = await client.responses.create(
            model=model, instructions=INSTRUCTIONS, input=history,
            tools=TOOLS, parallel_tool_calls=False, max_output_tokens=1000,
            store=False,
        )
        if response.status != "completed":
            raise AgentFailure("The model did not complete its response.")
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            if not response.output_text.strip():
                raise AgentFailure("The model returned no answer.")
            return response.output_text, used
        if len(calls) > 4:
            raise AgentFailure("Too many tool calls.")
        # Keep all output items, including reasoning items required by some models.
        history.extend(response.output)
        for call in calls:
            result = execute_tool(call.name, call.arguments)
            if "error" not in json.loads(result):
                used.append(call.name)
            history.append({"type": "function_call_output", "call_id": call.call_id,
                            "output": result})
    raise AgentFailure("The agent reached its step limit.")

## 6. Decision and limits

The tested flow demonstrates that valid requests reach a bounded agent loop and that tool requests pass an allowlist and argument validation. It does **not** establish the quality of live model answers, provider account access, or production readiness.

To run the actual model, configure the three variables in `.env.example` and follow the [README](README.md). Use `/docs` to send your first authenticated request. Keep keys on the server. The example has no database, tenant isolation or persistent conversation memory; the in-process limiter is for a single worker.

**Exercises:** add another read-only lesson topic; test missing arguments; replace lesson storage with an authorized database lookup; add an evaluation set for unsupported questions before publishing an API.

References: [FastAPI request bodies](https://fastapi.tiangolo.com/tutorial/body/), [OpenAI function calling](https://developers.openai.com/api/docs/guides/function-calling), [Pydantic models](https://docs.pydantic.dev/latest/concepts/models/).